In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Libraries loaded successfully!')

In [ ]:
ipums_path = os.path.expanduser('~/repoad688-employability-sum26-groupB/data/processed/employed_only.csv')
ipums = pd.read_csv(ipums_path)
print(f'Shape: {ipums.shape}')
print(f'\nColumns: {ipums.columns.tolist()}')
ipums.head(10)

In [ ]:
print('Summary Statistics:')
ipums.describe(include='all')

In [ ]:
missing = ipums.isnull().sum()
missing_pct = (missing / len(ipums) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)
print(f'Columns with missing values: {len(missing_df)} of {ipums.shape[1]}')
missing_df

In [ ]:
print('Gender Distribution:')
print(ipums['SEX_LABEL'].value_counts())
print()
print('Gender % Distribution:')
print((ipums['SEX_LABEL'].value_counts(normalize=True) * 100).round(2))

In [ ]:
zero_wage = ipums[ipums['INCWAGE'] == 0]
print(f'Records with $0 wage: {len(zero_wage):,}')
print(f'Percentage: {(len(zero_wage) / len(ipums) * 100):.2f}%')
print()
print('Gender breakdown of $0 wage:')
print(zero_wage['SEX_LABEL'].value_counts())

In [ ]:
print('Average wage INCLUDING $0:')
print(ipums.groupby('SEX_LABEL')['INCWAGE'].mean().round(2))

print('\nAverage wage EXCLUDING $0:')
ipums_nonzero = ipums[ipums['INCWAGE'] > 0]
print(ipums_nonzero.groupby('SEX_LABEL')['INCWAGE'].mean().round(2))
print(f'\nRows remaining after removing $0: {len(ipums_nonzero):,}')

In [ ]:
ipums_clean = ipums[ipums['INCWAGE'] > 0].copy()
print(f'Clean dataset shape: {ipums_clean.shape}')
print(f'Removed {len(ipums) - len(ipums_clean):,} rows with $0 wage')

In [ ]:
avg_wage = ipums_clean.groupby('SEX_LABEL')['INCWAGE'].mean().reset_index()

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(avg_wage['SEX_LABEL'], avg_wage['INCWAGE'],
              color=['tomato', 'steelblue'], edgecolor='black', width=0.5)

ax.set_title('Average Wage by Gender (Employed, 2024)', fontsize=14)
ax.set_xlabel('Gender')
ax.set_ylabel('Average Annual Wage ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

for bar, val in zip(bars, avg_wage['INCWAGE']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'${val:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
clean_path = os.path.expanduser('~/repoad688-employability-sum26-groupB/data/clean/')
os.makedirs(clean_path, exist_ok=True)

output_path = f'{clean_path}employed_only_clean.csv'
ipums_clean.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')
print(f'Shape: {ipums_clean.shape}')